In [57]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

In [58]:
db_people_file = Path(project_root / "data/from db/people-from-db.json")
people_file = Path(project_root / "data/people/prepped4loading/people.json")


with open(db_people_file, "r") as f:
   db_people = json.load(f)

with open(people_file, "r") as f:
   people = json.load(f)


db_lookup = {p["unified_id"]: p for p in db_people}
# rprint(dict(list(db_lookup.items())[:2]))

people_dict = {p["unified_id"]: p for p in people}
# rprint(dict(list(people_dict.items())[:2]))


In [59]:
people_fixed_ids = {}
name_changes = {}


name_fields = ["family_name", "given_names", "name_prefix", "name_particles", "name_suffix", "single_name", "is_organisation"]

for unified_id, person in people_dict.items():
# for unified_id, person in list(people_dict.items())[:1]:
    person_id_old = person["person_id"]
    family_name = person["family_name"]
    given_names = person["given_names"]
    name_prefix = person["name_prefix"]
    name_particles = person["name_particles"]
    name_suffix = person["name_suffix"]
    single_name = person["single_name"]
    is_organisation = person["is_organisation"]


    data = db_lookup[unified_id]
    person_id_new = data["person_id"]

    if any(person[f] != data[f] for f in name_fields):
        name_changes[unified_id] = {
            "json": person,
            "db": data
        }
    else:
        people_fixed_ids[unified_id] = {
            **person,
            "person_id": person_id_new,
            "person_id_old": person_id_old,
        }

# with open("name_updates.json", "w") as f:
#     json.dump(name_changes, f, ensure_ascii=False, indent=2)

fixing_path = Path(project_root / "data/people/fixing_ids")
fixed_file = Path(project_root / fixing_path / "people_id_mapping.json")

# with open(fixed_file, "w") as f:
#     json.dump(people_fixed_ids, f, ensure_ascii=False, indent=2)


# rprint(f"""
# === LOG ===
# total people: {len(people)}
# total db people: {len(db_people)}

# updates: {len(name_changes)}
# ids_fixed: {len(people_fixed_ids)}

# lets do some math - sum of updates + fixed:
# {len(name_changes) + len(people_fixed_ids)}
# === DONE ===
# """)

In [60]:
matches_file = Path(project_root / "data/people/fixing_ids/matches_ready.json")

with open(fixed_file, "r") as f:
   people = json.load(f)

with open(matches_file, "r") as f:
   matches = json.load(f)

# k = id old, value = person
people_id_dict = {str(v["person_id_old"]): v for k, v in people.items()}
# rprint(dict(list(people_id_dict.items())[:1]))

b2p_data = {}
b2p_list = []
problems = {}
seen_combinations = set()
duplicates = {}

for id_old, entries in matches.items():
# for id_old, entries in list(matches.items())[:3]:
   data = people_id_dict[id_old]
   person_id = data["person_id"]
   unified_id = data["unified_id"]

   for entry in entries:
      composite_id = entry["composite_id"]
      combo = (unified_id, composite_id)

      if combo in seen_combinations:
         duplicates[unified_id] = {
            "combo": combo,
            "entry": entry,
         }
      else:
         seen_combinations.add(combo)

         b2p_list.append({
            "person_id": person_id,
            "unified_id": unified_id,
            **entry})

      b2p_data[person_id] = {
         "person_id": person_id,
         "person_id_old": data["person_id_old"],
         "unified_id": unified_id,
         "entries": entries
      }



      # b2p_data[person_id] = {
      #    "person_id": person_id,
      #    "person_id_old": data["person_id_old"],
      #    "unified_id": unified_id,
      #    **entry}
   # if len(entries) == 1:
   #    b2p_list.append()


# rprint(dict(list(b2p_data.items())[:3]))
b2p_file = Path(project_root / fixing_path / "b2p_list.json")
b2p_data_file = Path(project_root / fixing_path / "b2p_data.json")
duplicates_file = Path(project_root / fixing_path / "duplicate_combos.json")

with open(duplicates_file, "w") as f:
    json.dump(duplicates, f, ensure_ascii=False, indent=2)

# with open(b2p_file, "w") as f:
#     json.dump(b2p_list, f, ensure_ascii=False, indent=2)

# with open(b2p_data_file, "w") as f:
#     json.dump(b2p_data, f, ensure_ascii=False, indent=2)


rprint(f"""
=== LOG ===
total matches: {len(matches)}
b2p_data len: {len(b2p_data)}
b2p_list len: {len(b2p_list)}
duplicates: {len(duplicates)}


=== DONE ===
""")


=== LOG ===
total matches: 1976
b2p_data len: 1976
b2p_list len: 2390
duplicates: 72


=== DONE ===